# OBJECTIVE

# Refactor the delivery-time prediction code built in P02/P03 from scattered notebook cells into a proper Python package (delivery/) with separate modules — data.py, features.py, model.py, and validate.py — tied together with an __init__.py. Import and use this package like a library to train and save a model, then run the full pipeline from the command line using standalone scripts (train.py and predict.py), without relying on a notebook.

# Tasks:

# T1: Add a function average_speed_kmph(distance_km, delivery_min) to the package.

# T2: Add a delivery/validate.py module with a function that rejects an impossible/unrealistic order.

# T3: Write a second command-line script, predict.py, that loads the saved model and predicts delivery time for a new order.

# Create Folders

In [1]:
#`delivery/` will hold our package files, `data/` will hold the CSV.

In [2]:
import os
os.makedirs("delivery", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("Folders ready")

Folders ready


# Step 1: data.py --- loads the data

In [3]:
%%writefile delivery/data.py
import pandas as pd
import numpy as np
import os

def load_data():
    path = "data/delivery_times.csv"

    if not os.path.exists(path):
        np.random.seed(42)
        n = 600
        distance_km = np.random.uniform(0.5, 12, n)
        prep_time_min = np.random.uniform(5, 30, n)
        traffic_level = np.random.randint(1, 4, n)
        rain = np.random.randint(0, 2, n)
        noise = np.random.normal(0, 2, n)

        delivery_min = 6 + 3 * distance_km + 0.6 * prep_time_min + 4 * traffic_level + 5 * rain + noise

        df = pd.DataFrame({
            "distance_km": distance_km,
            "prep_time_min": prep_time_min,
            "traffic_level": traffic_level,
            "rain": rain,
            "delivery_min": delivery_min
        })
        df.to_csv(path, index=False)

    return pd.read_csv(path)

Overwriting delivery/data.py


# Step 2: features.py --- splits X and y (also has T1)

# Splits the data into inputs (X) and the target to predict (y).
# average_speed_kmph is Task T1.

In [4]:
%%writefile delivery/features.py
def get_features_and_target(df):
    X = df[["distance_km", "prep_time_min", "traffic_level", "rain"]]
    y = df["delivery_min"]
    return X, y

# T1
def average_speed_kmph(distance_km, delivery_min):
    hours = delivery_min / 60
    return distance_km / hours

Overwriting delivery/features.py


# Step 3: model.py --- trains, checks, and saves the model

# Trains a LinearRegression model, prints its test error (MAE), and saves it to a file so predict.py can reuse it later.

In [5]:
%%writefile delivery/model.py
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

def train_and_save_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print("Test MAE:", round(mae, 2))

    joblib.dump(model, "delivery_model.joblib")
    return model

Overwriting delivery/model.py


# Step 4: validate.py --- T2, rejects a bad order
# Checks that an order's values are realistic before we predict on it.

In [6]:
%%writefile delivery/validate.py
# T2
def is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    if distance_km <= 0:
        return False
    if prep_time_min <= 0:
        return False
    if traffic_level not in [1, 2, 3]:
        return False
    if rain not in [0, 1]:
        return False
    return True

Overwriting delivery/validate.py


# Step 5: __init__.py --- makes delivery a package

# Built last so it can import functions from all the other files. This is what makes `from delivery import ...` work.

In [7]:
%%writefile delivery/__init__.py
from .data import load_data
from .features import get_features_and_target, average_speed_kmph
from .model import train_and_save_model
from .validate import is_valid_order

Overwriting delivery/__init__.py


# Step 6: Import and use our own package
# Proves the package works --- import it like a library and train the model.

In [8]:
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)

Test MAE: 1.92


# Step 7: train.py  script

In [9]:
%%writefile train.py
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)
print("Training done")

Overwriting train.py


# Step 8: Run train.py like a real program
# Runs the script as a terminal command, no notebook involved.

In [10]:
!python train.py

Test MAE: 1.92
Training done


# T3: predict.py --- second script
# Loads the saved model and predicts delivery time for one new order.

In [11]:
%%writefile predict.py
import sys
import joblib
import pandas as pd
from delivery import is_valid_order

distance_km = float(sys.argv[1])
prep_time_min = float(sys.argv[2])
traffic_level = int(sys.argv[3])
rain = int(sys.argv[4])

if not is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    print("Invalid order")
else:
    model = joblib.load("delivery_model.joblib")
    order = pd.DataFrame([[distance_km, prep_time_min, traffic_level, rain]],
                          columns=["distance_km", "prep_time_min", "traffic_level", "rain"])
    prediction = model.predict(order)[0]
    print("Predicted delivery time:", round(prediction, 1), "minutes")

Overwriting predict.py


# Test predict.py --- a normal order

In [12]:
!python predict.py 5.0 15 2 0

Predicted delivery time: 39.8 minutes


# Test predict.py --- a bad order (T2 check)

In [13]:
!python predict.py -3 15 2 0

Invalid order


# Test T1: average_speed_kmph

In [14]:
from delivery import average_speed_kmph

speed = average_speed_kmph(distance_km=5.0, delivery_min=40.4)
print("Average speed (km/h):", round(speed, 2))

Average speed (km/h): 7.43


# Summary 


- data.py --- loads the delivery data
- features.py --- splits data into X and y
- model.py --- trains, checks, and saves the model
- validate.py --- rejects a bad order
- __init__.py --- makes the folder a package

# To Do: Package the classification workflow

Using the same pattern from this lab (data.py -> features.py -> model.py ->
validate.py -> __init__.py), turn the classification code from Lab 3
(Breast Cancer dataset) into its own package. This time, go a step further
than just training one fixed model.

T1 --- model.py should not train just one model. Use cross-validation to
compare LogisticRegression, DecisionTreeClassifier, and
RandomForestClassifier, and automatically save whichever one scores best
(instead of hardcoding the winner yourself).

T2 --- validate.py should check that at least 3 of the input measurements
fall within a realistic range (e.g. radius_mean and area_mean can't be
negative or absurdly large) --- not just "is this a number".

T3 --- predict.py should print both the prediction (malignant/benign) AND
the model's confidence for that prediction (use predict_proba).

T4 --- add a metrics.py module with one function that prints a
confusion matrix and classification report for the saved model, so anyone
can check its performance without retraining.

In [15]:
import os

os.makedirs("delivery_cls", exist_ok=True)

In [ ]:
#data.py
%%writefile delivery_cls/data.py
import pandas as pd

def load_data():
    return pd.read_csv("data/breast-cancer.csv")

Overwriting delivery_cls/data.py


In [ ]:
#features.py
%%writefile delivery_cls/features.py
def get_features_and_target(df):
    X = df.drop(
        columns=["diagnosis", "id", "Unnamed: 32"],
        errors="ignore"
    )
    y = df["diagnosis"].map({"M": 1, "B": 0})

    return X, y

Overwriting delivery_cls/features.py


In [ ]:
#model.py
%%writefile delivery_cls/model.py
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def train_models(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    models = {
        "Logistic Regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=5000)
        ),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(
            n_estimators=50,
            random_state=42
        )
    }

    results = {}

    for name, model in models.items():
        model.fit(X_train, y_train)

        results[name] = [
            accuracy_score(y_train, model.predict(X_train)),
            accuracy_score(y_test, model.predict(X_test))
        ]

    return models, results, X_test, y_test

Overwriting delivery_cls/model.py


In [ ]:
#validate.py
%%writefile delivery_cls/validate.py
def is_valid_order(features):
    return all(value >= 0 for value in features)

Overwriting delivery_cls/validate.py


In [ ]:
#__init__.py
%%writefile delivery_cls/__init__.py
from .data import load_data
from .features import get_features_and_target
from .model import train_models
from .validate import is_valid_order

Overwriting delivery_cls/__init__.py


In [ ]:
#Run the package
from delivery_cls import load_data, get_features_and_target, train_models

df = load_data()
X, y = get_features_and_target(df)

models, results, X_test, y_test = train_models(X, y)

print(results)

{'Logistic Regression': [0.9868131868131869, 0.9649122807017544], 'Decision Tree': [1.0, 0.9298245614035088], 'Random Forest': [1.0, 0.9736842105263158]}


In [ ]:
#Create train_cls.py
%%writefile train_cls.py
from delivery_cls import load_data, get_features_and_target, train_models
import joblib

df = load_data()
X, y = get_features_and_target(df)

models, results, X_test, y_test = train_models(X, y)

best_model = max(results, key=lambda x: results[x][1])

joblib.dump(
    models[best_model],
    "classification_model.pkl"
)

print("Best Model:", best_model)
print("Test Accuracy:", results[best_model][1])

Writing train_cls.py


In [ ]:
#Run training script
!python train_cls.py

Best Model: Random Forest
Test Accuracy: 0.9736842105263158


In [ ]:
#Verify package
from delivery_cls import load_data, get_features_and_target

df = load_data()
X, y = get_features_and_target(df)

print("Dataset shape:", df.shape)
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Package working successfully")

Dataset shape: (569, 32)
Features shape: (569, 30)
Target shape: (569,)
Package working successfully
